# KTDB OD 예측 모델 학습 

In [1]:
from google.colab import drive
import os

drive.mount('/content/drive')

# 내 구글 드라이브의 KTDB/src/model 폴더로 이동 
os.chdir('/content/drive/MyDrive/kt/KTDB')

Mounted at /content/drive


### MAE 모델 학습

In [ ]:
!python src/model/train.py --model mae1 --epochs 50 --batch_size 32

### twostage 모델 학습

In [ ]:
!python src/twostage/train.py --epochs 50 --batch_size 32

---
### [experiment/loss-comparison] MAE1(SpatialODMAE1) Loss 탐색 자동 실험 (Colab GPU)

로컬 Mac MPS에서 1 epoch에 약 38분이 걸려(Colab T4 대비 약 17~19배 느림) Colab GPU로 전환함.
아래 셀들은 `scripts/run_mae1_loss_search.py`(신규 loss 5종 자동 screening 오케스트레이터)를 stage별로 나눠 실행함 — Colab 세션이 끊겨도 같은 `RUN_DIR`로 다시 실행하면 이미 끝난 조합은 건너뛰고 이어서 진행됨(skip-if-done).

**대상**: `src/model`의 `SpatialODMAE1`만. `src/twostage`, `src/gravity(경훈)`는 이 실험과 무관.

**실행 순서**: 패키지 설치 → RUN_DIR 설정 → target 분포 분석 → Stage1(baseline smoke) → Stage2(신규 loss smoke) → Stage3(10 epoch screening) → Stage4(50 epoch 최종) → 리포트 생성

#### 0-1. 필요 패키지 설치
Colab에 기본 설치돼 있지 않을 수 있는 패키지만 설치(torch/numpy/pandas/scikit-learn/matplotlib는 보통 이미 있음).

In [ ]:
!pip install -q lightgbm joblib

#### 0-2. RUN_DIR 설정
**새로 시작**: 아래 셀을 그대로 실행하면 새 타임스탬프 폴더가 생성됨.

**이어서 하기(세션이 끊겼다 재연결된 경우)**: 자동 생성 줄은 주석 처리하고, 이전에 출력됐던 RUN_DIR 값(예: artifacts/loss_search/20260716_090000)을 직접 지정한 뒤 실행.

이 폴더는 Drive에 저장되므로(1번 셀에서 이미 kt/KTDB로 이동한 상태) 세션이 끊겨도 결과는 보존됨.

In [ ]:
import time, os, sys

# 새로 시작: 아래 줄 사용
RUN_DIR = f"artifacts/loss_search/{time.strftime('%Y%m%d_%H%M%S')}"
# 이어서 하기: 위 줄을 주석 처리하고 아래처럼 이전 폴더를 직접 지정
# RUN_DIR = "artifacts/loss_search/20260716_090000"

os.makedirs(RUN_DIR, exist_ok=True)
os.environ['RUN_DIR'] = RUN_DIR
os.environ['PY_EXE'] = sys.executable
print('RUN_DIR =', RUN_DIR)
print('PY_EXE =', sys.executable)

#### 1. Target 분포 분석
학습 target(train_indices x train_indices OD 부분행렬)의 분포를 분석해 신규 loss의 데이터 기반 하이퍼파라미터(derived_loss_hparams.json)를 생성함. Stage 1 실행 전 반드시 1회 필요.

In [ ]:
!python scripts/analyze_target_distribution.py --out-dir "$RUN_DIR"

#### 2. Stage 1 — weighted_mse baseline smoke test
**여기서 지정한 epoch/seed/topk 등 설정이 experiment_config.json에 저장되어 이후 Stage 2~4 전체에 고정 적용됨** (이후 셀에서 값을 바꿔 넘겨도 무시되고 이 시점 설정이 유지됨 — 중간에 조건이 바뀌지 않도록 하기 위함). 실패하면 이후 stage는 자동으로 진행되지 않음.

In [ ]:
!python scripts/run_mae1_loss_search.py \
  --out-dir "$RUN_DIR" --python "$PY_EXE" --stage 1 \
  --model mae1 --batch-size 32 \
  --smoke-epochs 2 --screening-epochs 10 --final-epochs 50 \
  --smoke-seed 999 --main-seed 42 \
  --time-budget-hours 8 --final-topk 3 --final-topk-fallback 2

#### 3. Stage 2 — 신규 loss 5종 x 하이퍼파라미터 조합 smoke test (2 epoch)
실패한 조합은 최대 2회 재시도 후 제외되고, 다른 조합 실행은 계속됨.

In [ ]:
!python scripts/run_mae1_loss_search.py --out-dir "$RUN_DIR" --python "$PY_EXE" --stage 2

#### 4. Stage 3 — 10 epoch screening (baseline + Stage2 통과 후보)
screening_leaderboard.csv 생성, 안전조건(작은 OD MAE 10%+ 악화 제외) 적용 후 상위 후보 선정.

In [ ]:
!python scripts/run_mae1_loss_search.py --out-dir "$RUN_DIR" --python "$PY_EXE" --stage 3

#### 5. Stage 4 — 50 epoch 최종 실험 (baseline + Stage3 상위 후보)
예상 총 시간이 --time-budget-hours(기본 8시간)를 넘으면 후보를 자동으로 축소함. final_leaderboard.csv 생성.

In [ ]:
!python scripts/run_mae1_loss_search.py --out-dir "$RUN_DIR" --python "$PY_EXE" --stage 4

#### 6. 최종 리포트 생성
final_report.md(환경/수식/하이퍼파라미터 근거/leaderboard/추천 loss 등 19개 항목)를 생성함. 위 stage 어디서든 다시 실행해도 안전(디스크에 남은 결과로부터 재구성).

In [ ]:
!python scripts/run_mae1_loss_search.py --out-dir "$RUN_DIR" --python "$PY_EXE" --stage report

#### 결과 확인 경로
```
$RUN_DIR/RUN_STATUS.md          # 진행 로그(각 단계 완료 시점마다 갱신)
$RUN_DIR/target_distribution.md # target 분포 분석
$RUN_DIR/screening_leaderboard.csv
$RUN_DIR/final_leaderboard.csv
$RUN_DIR/final_report.md        # 최종 추천 loss와 근거
$RUN_DIR/runs/<stage>_<loss>_.../train.log, results.csv, visualization.png, best_model.pth
```
세션이 끊겼다면: 이 노트북을 다시 열고 0-2번 셀에서 RUN_DIR을 위 경로로 지정한 뒤, 중단됐던 단계부터(또는 처음부터, 이미 끝난 조합은 자동으로 건너뜀) 다시 실행하면 됨.